# 00_utilidades

Este notebook **no se ejecuta solo**: los demás notebooks lo llaman con `%run ./00_utilidades`.
Aquí viven las rutas, los nombres de tablas y las funciones que todos comparten, para que
cada integrante use exactamente la misma configuración y la misma limpieza.

In [ ]:
import time, statistics, unicodedata
from pyspark.sql import functions as F, Row
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

# ---- Dónde está todo en Unity Catalog ----
CATALOGO = "workspace"
ESQUEMA  = "experimentobigdata"
VOLUMEN  = "registro_nacidos"
RUTA_VOL    = f"/Volumes/{CATALOGO}/{ESQUEMA}/{VOLUMEN}"
RUTA_CSV    = f"{RUTA_VOL}/dataSet/DataSet Registro Nacidos.CSV"   # copia exacta el nombre que ves en Catalog
RUTA_EXPORT = f"{RUTA_VOL}/export_sql"                              # aquí quedan los CSV para SQL Server

SEMILLA = 42              # número inicial del generador aleatorio (ver explicación en el notebook 04)
FORMATO_TABLAS = "delta"  # formato de las tablas que guardamos

# ---- Nombres de las tablas (arquitectura Medallion) ----
T_BRONZE     = "bronze_nacimientos"        # CSV tal cual llegó
T_SILVER_1X  = "silver_nacimientos_1x"     # limpio, datos reales (1,048,575)
T_SILVER_5X  = "silver_nacimientos_5x"     # real + 4 millones sintéticos
T_SILVER_10X = "silver_nacimientos_10x"    # real + 9 millones sintéticos
T_SINTETICO  = "bronze_sintetico"          # datos ficticios generados con Faker
T_RESULTADOS = "resultados_experimentos"   # tabla común donde todos guardan sus tiempos

FILAS_SINT_5X  = 4_000_000
FILAS_SINT_10X = 9_000_000

In [ ]:
def nombre_tabla(nombre):
    """Devuelve catalogo.esquema.tabla."""
    return f"{CATALOGO}.{ESQUEMA}.{nombre}" if CATALOGO else f"{ESQUEMA}.{nombre}"

def leer_tabla(nombre):
    return spark.table(nombre_tabla(nombre))

def guardar_tabla(df, nombre, particion=None):
    """Guarda un DataFrame como tabla (sobrescribe). Si se indica, particiona por esa columna."""
    w = df.write.mode("overwrite").format(FORMATO_TABLAS).option("overwriteSchema", "true")
    if particion:
        w = w.partitionBy(particion)
    w.saveAsTable(nombre_tabla(nombre))

In [ ]:
# Las 22 columnas del CSV del MINSA, en su orden original
COLUMNAS_ORIGINALES = [
    "FecNac_Año", "FecNac_Mes", "PESO_NACIDO", "TALLA_NACIDO", "DUR_EMB_PARTO", "Condicion_Parto",
    "sexo_nacido", "Tipo_Parto", "Edad_Madre", "Estado_Civil", "Nivel_Intrucción_Madre",
    "DESC_OCUPACION", "Num_embar_madre", "Hijos_vivo_madre", "Hijos_fallec_madre",
    "nacmuer_abort_madre", "Pais_Madre", "IdUbigeoInei", "Ipress", "Lugar_Nacido",
    "Atiende_Parto", "Financiador_Parto",
]

# Nombre nuevo (sin tildes ni mayúsculas raras) para cada columna: se usa con withColumnRenamed
MAPA_RENOMBRE = {
    "fecnac_ano": "anio", "fecnac_mes": "mes", "peso_nacido": "peso_gramos",
    "talla_nacido": "talla_cm", "dur_emb_parto": "semanas_gestacion",
    "condicion_parto": "condicion_parto", "sexo_nacido": "sexo", "tipo_parto": "tipo_parto",
    "edad_madre": "edad_madre", "estado_civil": "estado_civil",
    "nivel_intruccion_madre": "nivel_instruccion", "desc_ocupacion": "ocupacion",
    "num_embar_madre": "num_embarazos", "hijos_vivo_madre": "hijos_vivos",
    "hijos_fallec_madre": "hijos_fallecidos", "nacmuer_abort_madre": "nacidos_muertos_abortos",
    "pais_madre": "pais_madre", "idubigeoinei": "ubigeo", "ipress": "ipress",
    "lugar_nacido": "lugar_nacido", "atiende_parto": "atiende_parto",
    "financiador_parto": "financiador_parto",
}

def _ascii(texto):
    """'FecNac_Año' -> 'fecnac_ano' (quita tildes y pasa a minúsculas)."""
    return unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode().lower()

NOMBRE_SILVER = {c: MAPA_RENOMBRE[_ascii(c)] for c in COLUMNAS_ORIGINALES}

COLUMNAS_NUMERICAS = {"anio": "int", "mes": "int", "peso_gramos": "double", "talla_cm": "double",
                      "semanas_gestacion": "int", "edad_madre": "int"}
FINANCIADORES_VALIDOS = ["SIS", "ESSALUD", "PARTICULAR", "PRIVADOS", "SANIDAD EP",
                         "SANIDAD FAP", "SANIDAD NAVAL", "SANIDAD PNP", "IGNORADO"]

In [ ]:
def limpiar_a_silver(df):
    """Bronze (22 columnas de texto) -> Silver (columnas renombradas, tipadas y validadas).
    Usa las mismas reglas que el ETL de SQL Server para poder comparar resultados."""
    # 1) withColumnRenamed + select: nombres nuevos y solo las 22 columnas de negocio
    for c in df.columns:
        k = _ascii(c)
        if k in MAPA_RENOMBRE:
            df = df.withColumnRenamed(c, MAPA_RENOMBRE[k])
    nombres = list(MAPA_RENOMBRE.values())
    df = df.select(*nombres)

    # 2) Texto: quitar comillas, ';' y saltos de línea, recortar espacios y convertir "" en nulo
    def limpio(c):
        t = F.trim(F.regexp_replace(F.col(c), r'[";\r\n]', ""))
        return F.when(t == "", F.lit(None)).otherwise(t)
    df = df.select(*[limpio(c).alias(c) for c in nombres])

    # 3) Tipado defensivo: try_cast devuelve nulo (no error) si el texto no es un número
    df = df.select(*[F.expr(f"try_cast(`{c}` as {COLUMNAS_NUMERICAS[c]})").alias(c)
                     if c in COLUMNAS_NUMERICAS else F.col(c) for c in nombres])

    # 4) filter: descartar filas completamente vacías (igual que el ETL en SQL)
    df = df.filter(F.col("anio").isNotNull() | F.col("peso_gramos").isNotNull() | F.col("edad_madre").isNotNull())

    # 5) Validez: valores imposibles pasan a nulo
    df = (df
          .withColumn("peso_gramos", F.when(F.col("peso_gramos") > 0, F.col("peso_gramos")))
          .withColumn("talla_cm", F.when(F.col("talla_cm").between(20, 70), F.col("talla_cm")))
          .withColumn("semanas_gestacion", F.when(F.col("semanas_gestacion").between(20, 45), F.col("semanas_gestacion")))
          .withColumn("edad_madre", F.when(F.col("edad_madre").between(10, 55), F.col("edad_madre"))))

    # 6) Categorías en mayúsculas y reglas de negocio (filas desplazadas del CSV)
    categorias = [c for c in nombres if c not in COLUMNAS_NUMERICAS]
    for c in categorias:
        df = df.withColumn(c, F.upper(F.col(c)))
    solo_digitos = lambda c: F.col(c).rlike("^[0-9]+$")
    df = (df
          .withColumn("pais_madre", F.when(F.col("pais_madre") == "1", "PERU")
                                     .when(F.col("pais_madre") == "2", "EXTRANJERO")
                                     .when(solo_digitos("pais_madre"), "IGNORADO")
                                     .otherwise(F.col("pais_madre")))
          .withColumn("lugar_nacido", F.when(solo_digitos("lugar_nacido"), "ESTABLECIMIENTO DE SALUD")
                                       .otherwise(F.col("lugar_nacido")))
          .withColumn("atiende_parto", F.when(solo_digitos("atiende_parto") |
                                              F.col("atiende_parto").isin("ESTABLECIMIENTO DE SALUD", "PERU", "VENEZUELA"), None)
                                        .otherwise(F.col("atiende_parto")))
          .withColumn("financiador_parto", F.when(F.col("financiador_parto").isin(*FINANCIADORES_VALIDOS),
                                                  F.col("financiador_parto")))
          .withColumn("num_embarazos", F.when(F.col("num_embarazos").rlike(r"^(-?[0-9]+|>=5)$"),
                                              F.col("num_embarazos"))))

    # 7) na.fill: lo que quedó sin valor en categorías pasa a 'IGNORADO'; año/mes sin valor pasan a -1
    df = df.na.fill("IGNORADO", subset=categorias).na.fill({"anio": -1, "mes": -1})

    # 8) Indicadores por fila (mismas definiciones que el ETL de SQL Server)
    df = (df
          .withColumn("ind_bajo_peso", F.when((F.col("peso_gramos") > 0) & (F.col("peso_gramos") < 2500), 1).otherwise(0))
          .withColumn("ind_madre_adolesc", F.when(F.col("edad_madre").between(10, 19), 1).otherwise(0))
          .withColumn("grupo_edad", F.when(F.col("edad_madre").isNull(), "IGNORADO")
                                     .when(F.col("edad_madre").between(10, 19), "ADOLESCENTE")
                                     .otherwise("ADULTA")))
    return df

In [ ]:
def calcular_kpis(df, *grupo):
    """I1 total de nacimientos | I2 % bajo peso (sobre nacidos con peso válido) | I3 % madres adolescentes.
    Misma definición que las medidas DAX del dashboard de Power BI."""
    r = df.groupBy(*grupo).agg(
        F.count("*").alias("i1_total_nacimientos"),
        F.sum("ind_bajo_peso").alias("n_bajo_peso"),
        F.sum(F.when(F.col("peso_gramos") > 0, 1).otherwise(0)).alias("n_con_peso"),
        F.sum("ind_madre_adolesc").alias("n_madres_adolescentes"))
    return (r
            .withColumn("i2_pct_bajo_peso",
                        F.when(F.col("n_con_peso") > 0, F.round(F.col("n_bajo_peso") / F.col("n_con_peso") * 100, 2)))
            .withColumn("i3_pct_madres_adolescentes",
                        F.round(F.col("n_madres_adolescentes") / F.col("i1_total_nacimientos") * 100, 2)))

In [ ]:
def medir(accion, repeticiones=3, calentamientos=1):
    """Ejecuta 'accion' (una función sin argumentos) y devuelve la lista de tiempos en segundos.
    El calentamiento no se cuenta: la primera ejecución siempre es más lenta (arranque, caché de disco)."""
    for _ in range(calentamientos):
        accion()
    tiempos = []
    for _ in range(repeticiones):
        t0 = time.perf_counter()
        accion()
        tiempos.append(time.perf_counter() - t0)
    return tiempos

ESQUEMA_RESULTADOS = StructType([
    StructField("experimento", StringType(), False), StructField("variante", StringType(), False),
    StructField("escala", StringType(), True), StructField("filas", LongType(), True),
    StructField("media_s", DoubleType(), False), StructField("desv_s", DoubleType(), False),
    StructField("throughput_filas_s", LongType(), True), StructField("entorno", StringType(), False),
    StructField("responsable", StringType(), False), StructField("detalle", StringType(), True),
])

def registrar_resultado(experimento, variante, escala, filas, tiempos, entorno, responsable, detalle=""):
    """Guarda una fila en la tabla común de resultados (reemplaza la anterior si repites el experimento)."""
    media = statistics.mean(tiempos)
    desv = statistics.stdev(tiempos) if len(tiempos) > 1 else 0.0
    fila = Row(experimento=experimento, variante=variante, escala=escala,
               filas=int(filas) if filas else None, media_s=round(media, 3), desv_s=round(desv, 3),
               throughput_filas_s=int(filas / media) if filas else None,
               entorno=entorno, responsable=responsable, detalle=detalle)
    tabla = nombre_tabla(T_RESULTADOS)
    if spark.catalog.tableExists(tabla):
        spark.sql(f"DELETE FROM {tabla} WHERE experimento = '{experimento}' AND variante = '{variante}' "
                  f"AND COALESCE(escala, '') = '{escala or ''}'")
    spark.createDataFrame([fila], schema=ESQUEMA_RESULTADOS).write.mode("append").format(FORMATO_TABLAS).saveAsTable(tabla)
    return fila

In [ ]:
print("Utilidades cargadas. Tablas en:", f"{CATALOGO}.{ESQUEMA}")